# Simulating a Field of Stars — Liger Imager

In [1]:
from liger_iris_sim.sources import make_point_source_image
from liger_iris_sim.expose import compute_liger_throughput, expose_imager
from liger_iris_sim.utils import LIGER_PROPS, rebin_image, compute_filter_photon_flux

from liger_iris_drp_resources.filters import load_filters_summary
from liger_iris_drp_resources.psfs import get_liger_psf, download_liger_psfs
from liger_iris_drp_resources.model_spectra import download_model_spectra

import numpy as np
import matplotlib.pyplot as plt

## Download required resources

PSF files are downloaded automatically to the local resources directory on first use.

In [2]:
download_liger_psfs()
download_model_spectra()

'/Users/cale/.astropy/cache/download/url/LIGER_IRIS_DRP_RESOURCES/Model_Spectra'

## Instrument and exposure parameters

In [3]:
mode = 'img'          # imager mode
filt = 'J'
size = (2048, 2048)     # detector size in pixels
read_noise = 9        # e- RMS
dark_current = 0.025  # e- / sec / pixel
scale = 0.01          # arcsec / pixel
itime = 300           # integration time, sec
n_frames = 1          # number of coadded frames
collarea = LIGER_PROPS['keck_collarea']  # m^2

## Load filter data and compute throughput

`load_filters_summary` returns filter metadata including the central wavelength, zero-point flux, and background magnitude.
`compute_liger_throughput` folds together telescope, AO, filter, and instrument throughput.

In [4]:
filter_info = load_filters_summary(filter_name=filt)

tput = compute_liger_throughput(
    mode=mode,
    wave=filter_info['wavecenter'],
)
print(f"Total throughput for {mode} mode at {filter_info['wavecenter']:.3f} µm: {tput:.3f}")

Total throughput for img mode at 1.248 µm: 0.343


## Load and rebin the on-axis PSF

`get_liger_psf` returns the PSF at a given wavelength and field position.
`rebin_image` resamples the PSF from its native sampling to the detector pixel scale.

In [5]:
psf, psf_info = get_liger_psf(filter_info['wavecenter'], xs=0, ys=0)
psf = rebin_image(
    psf,
    scale_in=psf_info['psf_sampling'],
    scale_out=scale,
)
print(f"PSF shape after rebinning: {psf.shape}")

PSF shape after rebinning: (112, 112)


## Create a field of stars

`compute_filter_photon_flux` converts a magnitude to photons / sec / m² using the filter zero-point.
`make_point_source_image` places each star into the source rate image, convolved with the PSF.

In [6]:
np.random.seed(1)
n_stars = 10
mag_range = (18, 19)

source_rate = np.zeros(size, dtype=np.float32)

for i in range(n_stars):
    mag = np.random.uniform(mag_range[0], mag_range[1])
    xpix = int(np.random.uniform(5, size[1] - 5))
    ypix = int(np.random.uniform(5, size[0] - 5))
    photon_flux = compute_filter_photon_flux(mag, zp=filter_info['zpphot'])
    print(f"Star {i+1:2d}: mag={mag:.2f}, x={xpix}, y={ypix}, flux={photon_flux:.2e} phot/s/m²")
    make_point_source_image(
        xpix, ypix, photon_flux,
        psf=psf,
        image_out=source_rate,
    )

Star  1: mag=18.42, x=1473, y=5, flux=1.96e+02 phot/s/m²
Creating point source for xdet=1473, ydet=5, photon_flux=196.1521219330923 phot / sec / m^2
Star  2: mag=18.30, x=304, y=193, flux=2.18e+02 phot/s/m²
Creating point source for xdet=304, ydet=193, photon_flux=218.00616913024794 phot / sec / m^2
Star  3: mag=18.19, x=709, y=813, flux=2.43e+02 phot/s/m²
Creating point source for xdet=709, ydet=813, photon_flux=242.60387195948687 phot / sec / m^2
Star  4: mag=18.54, x=859, y=1401, flux=1.75e+02 phot/s/m²
Creating point source for xdet=859, ydet=1401, photon_flux=175.3376247043176 phot / sec / m^2
Star  5: mag=18.20, x=1794, y=60, flux=2.39e+02 phot/s/m²
Creating point source for xdet=1794, ydet=60, photon_flux=238.57279171494514 phot / sec / m^2
Star  6: mag=18.67, x=855, y=1143, flux=1.55e+02 phot/s/m²
Creating point source for xdet=855, ydet=1143, photon_flux=155.315503322124 phot / sec / m^2
Star  7: mag=18.14, x=408, y=1636, flux=2.53e+02 phot/s/m²
Creating point source for xdet=

## Sky background

The per-pixel sky emission rate is derived from the filter background magnitude and integrated over the pixel solid angle.

In [7]:
# photons / sec / arcsec^2 / m^2
sky_em_rate = compute_filter_photon_flux(filter_info['backmag'], zp=filter_info['zpphot'])
# Integrate over pixel solid angle -> photons / sec / m^2 / pixel
sky_em_rate *= scale**2
print(f"Sky emission rate: {sky_em_rate:.4e} phot/s/m²/pixel")

Sky emission rate: 1.0998e-02 phot/s/m²/pixel


## Run the exposure simulation

`expose_imager` adds Poisson noise, dark current, and read noise, and returns a dict of simulated images and noise components.

In [8]:
sim = expose_imager(
    source_rate,
    itime=itime, n_frames=n_frames, collarea=collarea,
    sky_emission_rate=sky_em_rate,
    tput=tput, read_noise=read_noise, dark_current=dark_current,
)
print(f"Peak SNR: {np.nanmax(sim['snr']):.1f}")

Peak SNR: 238.8


## Visualize results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rate_map = sim['observed_rate'].copy()
rate_map[rate_map <= 0] = np.nan  # Mask non-positive values for
im0 = axes[0].imshow(
    np.log10(rate_map),
    origin='lower', cmap='inferno',
)
axes[0].set_title('log10(Observed rate (e-/s))')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(sim['snr'], origin='lower', cmap='viridis', vmin=0)
axes[1].set_title('SNR per pixel')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()